# Product Hunt Synthetic Launch Analysis

Analyzing pre-launch factors (timing, presentation, team size, category) that correlate with launch success on Product Hunt. We now use the synthetic dataset (`synthetic_producthunt_10000.csv`) and preprocess it first so the rest of the notebook can reuse the same feature names.

## 1. Imports, Load, and Preprocess

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import xgboost as xgb

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

In [3]:
data_path = '/Users/kireeti/Desktop/Projects/Product hunt/synthetic_producthunt_10000.csv'
df = pd.read_csv(data_path)

# Normalize the synthetic schema and add compatibility columns for the rest of the notebook.
df.columns = df.columns.str.strip()
df = df.rename(columns={
    'votesCount': 'upvotes',
    'commentsCount': 'comments',
})

base_date = pd.Timestamp('2026-01-05')  # Monday
df['launch_timestamp'] = (
    base_date
    + pd.to_timedelta(df['weekday'].fillna(0).astype(int), unit='D')
    + pd.to_timedelta(df['launch_hour'].fillna(12).astype(int), unit='h')
).dt.strftime('%Y-%m-%d %H:%M:%S')
df['launch_datetime'] = pd.to_datetime(df['launch_timestamp'], format='mixed', errors='coerce', utc=True)
df['launch_day_name'] = df['launch_datetime'].dt.day_name()

df['team_size'] = df['maker_count'].fillna(0).astype(int)
df['categories'] = df['primary_topic'].fillna('Unknown').apply(lambda topic: str([topic]))
df['period'] = df['month'].fillna(1).astype(int).map(lambda month: f'2026-{month:02d}')
df['url'] = 'https://www.producthunt.com/posts/synthetic-launch-' + df.index.astype(str)
df['website_url'] = df['url']
df['thumbnail'] = pd.NA
df['mode'] = 'launch'
df['product_domain'] = (
    df['primary_topic']
    .fillna('unknown')
    .astype(str)
    .str.lower()
    .str.replace(r'[^a-z0-9]+', '-', regex=True)
    .str.strip('-')
)
df['maker_profile_urls'] = pd.NA
df['hunter_name'] = pd.NA
df['hunter_profile_url'] = pd.NA

print(f"Rows: {df.shape[0]}, Cols: {df.shape[1]}")
df.head()

Rows: 10000, Cols: 31


,rank,upvotes,comments,launch_hour,weekday,month,is_featured,description_length,tagline_length,topic_count,...,categories,period,url,website_url,thumbnail,mode,product_domain,maker_profile_urls,hunter_name,hunter_profile_url
0,3,10.0,96.0,6,1,5,True,73,45,3,...,['Open Source'],2026-05,https://www.producthunt.com/posts/synthetic-la...,https://www.producthunt.com/posts/synthetic-la...,<NA>,launch,open-source,<NA>,<NA>,<NA>
1,17,32.0,39.0,16,6,6,False,388,52,3,...,['iOS'],2026-06,https://www.producthunt.com/posts/synthetic-la...,https://www.producthunt.com/posts/synthetic-la...,<NA>,launch,ios,<NA>,<NA>,<NA>
2,9,88.0,6.0,7,3,6,True,500,56,3,...,['Productivity'],2026-06,https://www.producthunt.com/posts/synthetic-la...,https://www.producthunt.com/posts/synthetic-la...,<NA>,launch,productivity,<NA>,<NA>,<NA>
3,1,0.0,7.0,21,0,6,False,373,49,3,...,['Design Tools'],2026-06,https://www.producthunt.com/posts/synthetic-la...,https://www.producthunt.com/posts/synthetic-la...,<NA>,launch,design-tools,<NA>,<NA>,<NA>
4,1,117.0,6.0,8,0,5,True,85,38,3,...,['Productivity'],2026-05,https://www.producthunt.com/posts/synthetic-la...,https://www.producthunt.com/posts/synthetic-la...,<NA>,launch,productivity,<NA>,<NA>,<NA>


## 2. Data Limitations & Quality check

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype              
---  ------                  --------------  -----              
 0   rank                    10000 non-null  int64              
 1   upvotes                 10000 non-null  float64            
 2   comments                10000 non-null  float64            
 3   launch_hour             10000 non-null  int64              
 4   weekday                 10000 non-null  int64              
 5   month                   10000 non-null  int64              
 6   is_featured             10000 non-null  bool               
 7   description_length      10000 non-null  int64              
 8   tagline_length          10000 non-null  int64              
 9   topic_count             10000 non-null  int64              
 10  maker_count             10000 non-null  int64              
 11  maker_total_followers   10000 non-null  in

In [5]:
# Check missing values for hunter columns
df[['hunter_name', 'hunter_profile_url', 'self_launched']].isnull().sum()

hunter_name           10000
hunter_profile_url    10000
self_launched             0
dtype: int64

**Note:** `hunter_name` and `hunter_profile_url` are intentionally blank in the synthetic dataset, so the notebook keeps the same analysis flow while focusing on timing, media count, team size, and categories instead.